<a href="https://colab.research.google.com/github/SadakatSarin/Research-498R/blob/main/02_paper-review/MPA%3A%20Multimodal%20Prototype%20Augmentation%20for%20Few-Shot%20Learning/MPA_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 ***Instead of needing a paid OpenAI API key to call GPT-4.0 live during training, this self-contained Kaggle notebook uses CLIP (openai/clip-vit-base-patch32 or openai/clip-vit-large-patch14) via transformers (already pre-installed on Kaggle) and implements all three MPA modules (LMSE multi-variant text descriptions, HMA multi-view crops/flips/jitter, and AUCA prototype interpolation + Gaussian noise absorption) on CIFAR-FS:***

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from transformers import CLIPProcessor, CLIPModel
from sklearn.linear_model import LogisticRegression

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Running on:", device)

# 1. Load Frozen CLIP Backbone (switch to 'openai/clip-vit-large-patch14' to match the paper's exact backbone)
model_id = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(model_id).to(device).eval()
processor = CLIPProcessor.from_pretrained(model_id)

# Helper to unwrap tensor safely across transformers v4 & v5+
def unwrap_features(out):
    return out.pooler_output if hasattr(out, "pooler_output") else out

# 2. Load CIFAR-100 & Split into CIFAR-FS (Classes 64-99 = Unseen Test Classes)
raw_test = datasets.CIFAR100(root='/kaggle/working/data', train=False, download=True)
class_names = raw_test.classes

test_images_by_class = {}
for img, label in raw_test:
    if label >= 64:
        test_images_by_class.setdefault(label, []).append(img)

# 3. Module 1: LMSE (Multi-Variant Semantic Descriptions per Class)
def get_lmse_features(cls_id):
    name = class_names[cls_id].replace('_', ' ')
    variants = [
        f"a photo of a {name}, a type of object or creature",
        f"a clear close-up image showing the visual appearance of a {name}",
        f"a natural photograph capturing the shape, color, and texture of a {name}",
        f"an outdoor or indoor scene featuring a {name}",
        f"a detailed view of a {name} in its typical environment"
    ]
    inputs = processor(text=variants, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        text_feats = unwrap_features(clip_model.get_text_features(**inputs))
        text_feats = F.normalize(text_feats, dim=-1)
    return text_feats  # [5, d]

# 4. Module 2: HMA (Hierarchical Multi-View Augmentation: Naturalistic + Geometric)
hma_transforms = [
    transforms.RandomResizedCrop(32, scale=(0.7, 1.0)),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomHorizontalFlip(p=1.0),  # Geometric reflection view
]

def get_hma_features(pil_imgs):
    all_views = list(pil_imgs)  # Original views
    for img in pil_imgs:
        for t in hma_transforms:
            all_views.append(t(img))
    inputs = processor(images=all_views, return_tensors="pt").to(device)
    with torch.no_grad():
        img_feats = unwrap_features(clip_model.get_image_features(**inputs))
        img_feats = F.normalize(img_feats, dim=-1)
    return img_feats  # [num_imgs * 5, d]

# 5. Module 3: AUCA (Adaptive Uncertain Class Absorber via Interpolation + Gaussian Noise)
def generate_auca_samples(class_prototypes, num_samples=10):
    C = F.normalize(class_prototypes, dim=-1)  # [N_way, d]
    S = torch.mm(C, C.t())
    upper_tri = S[torch.triu(torch.ones_like(S), diagonal=1) == 1]

    if upper_tri.numel() > 1 and (upper_tri.max() - upper_tri.min()) > 1e-6:
        S_norm = (upper_tri - upper_tri.min()) / (upper_tri.max() - upper_tri.min())
        lam = 1.0 - S_norm.mean().item()
    else:
        lam = 0.5

    uncertain_feats = []
    n_way, dim = C.shape
    for _ in range(num_samples):
        if random.random() < lam:
            j, k = random.sample(range(n_way), 2)
            alpha = random.uniform(0.2, 0.8)
            sample = alpha * C[j] + (1 - alpha) * C[k]
        else:
            sample = torch.randn(dim, device=device)
        uncertain_feats.append(F.normalize(sample, dim=-1))
    return torch.stack(uncertain_feats)  # [num_samples, d]

# 6. Evaluate 5-Way 1-Shot Episodes on Unseen CIFAR-FS Classes
def evaluate_mpa(num_episodes=100, n_way=5, k_shot=1, q_query=15):
    accs = []
    for ep in range(num_episodes):
        selected = random.sample(list(test_images_by_class.keys()), n_way)
        X_train, y_train, prototypes = [], [], []
        X_test, y_test = [], []

        for idx, cls_id in enumerate(selected):
            imgs = random.sample(test_images_by_class[cls_id], k_shot + q_query)
            supp_imgs, query_imgs = imgs[:k_shot], imgs[k_shot:]

            # Extract HMA visual features + LMSE text features
            vis_feats = get_hma_features(supp_imgs)
            txt_feats = get_lmse_features(cls_id)
            combined_feats = torch.cat([vis_feats, txt_feats], dim=0)

            X_train.append(combined_feats)
            y_train.extend([idx] * combined_feats.size(0))
            prototypes.append(combined_feats.mean(dim=0))

            # Query features (original image only)
            q_inputs = processor(images=query_imgs, return_tensors="pt").to(device)
            with torch.no_grad():
                q_feats = F.normalize(unwrap_features(clip_model.get_image_features(**q_inputs)), dim=-1)
            X_test.append(q_feats)
            y_test.extend([idx] * q_query)

        # Add AUCA synthetic "Uncertain Class" (label = n_way)
        auca_feats = generate_auca_samples(torch.stack(prototypes), num_samples=10)
        X_train.append(auca_feats)
        y_train.extend([n_way] * auca_feats.size(0))

        # Fit Logistic Regression on enriched support set (Algorithm 1)
        clf = LogisticRegression(max_iter=1000, C=1.0)
        clf.fit(torch.cat(X_train).cpu().numpy(), np.array(y_train))

        # Predict among the n_way real classes (ignore uncertain class column at test time)
        probs = clf.predict_proba(torch.cat(X_test).cpu().numpy())[:, :n_way]
        preds = probs.argmax(axis=1)
        accs.append((preds == np.array(y_test)).mean())

    print(f"MPA 5-Way 1-Shot Accuracy on CIFAR-FS ({num_episodes} episodes): {np.mean(accs)*100:.2f}%")

evaluate_mpa(num_episodes=100)

Running on: cuda


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

MPA 5-Way 1-Shot Accuracy on CIFAR-FS (100 episodes): 87.28%


**87.28% 5-way 1-shot accuracy on CIFAR-FS on our very first run—that is a massive leap over traditional image-only few-shot baselines (which typically score ~50–65% on CIFAR-FS in 1-shot) and lines up right with the paper's backbone ablation (Table 7, where ViT-B/32 scores ~74–91% across benchmarks).**